In [3]:
import cv2
import numpy as np

def deskew_plate(plate):
    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    coords = np.column_stack(np.where(edges > 0))
    angle = cv2.minAreaRect(coords)[-1]

    # xử lý góc
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = plate.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(plate, M, (w, h),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)

    return rotated

In [4]:
def sort_ocr_text(result_ocr):
    combined = []

    for res in result_ocr:

        # lấy text + polygon
        texts = res.get('rec_texts', [])
        polys = res.get('rec_polys', [])

        for text, poly in zip(texts, polys):

            # lấy tọa độ góc trái trên
            x = poly[0][0]
            y = poly[0][1]

            combined.append((y, x, text))

    # ===== sort =====
    # ưu tiên theo y trước (dòng)
    # rồi theo x (trái -> phải)
    combined.sort(key=lambda item: (item[0] // 20, item[1]))

    final_text = " ".join([item[2] for item in combined])

    return final_text

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR
import cv2

model = YOLO("../model/best_plate_detection.pt")
ocr = PaddleOCR(use_angle_cls=True, lang='en')

cap = cv2.VideoCapture("../samples/video.mp4")
frame_id =0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1
    if frame_id % 5 != 0:
        continue
    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            plate = frame[y1:y2, x1:x2]
            plate = cv2.resize(plate, None, fx=2, fy=2)

            if plate.shape[0] != 0 and plate.shape[1] != 0:
                plate = deskew_plate(plate)

            result_ocr = ocr.predict(plate)

            # text = ""
            # if result_ocr and len(result_ocr) > 0:
            #     for res in result_ocr:
            #         if 'rec_texts' in res:
            #             text += " ".join(res['rec_texts'])
            text = sort_ocr_text(result_ocr)
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

            cv2.putText(frame, text.strip(), (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, (0,255,0), 2)

    cv2.imshow("Video", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

/var/folders/5l/xvvkbgln1qz502f2b5fvbfj40000gn/T/ipykernel_2458/2556991964.py:6: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en')
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/vi/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/vi/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/vi/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files.


0: 544x960 2 plates, 162.0ms
Speed: 3.3ms preprocess, 162.0ms inference, 0.9ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 198.8ms
Speed: 2.9ms preprocess, 198.8ms inference, 1.0ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 143.6ms
Speed: 2.6ms preprocess, 143.6ms inference, 0.7ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 274.8ms
Speed: 4.4ms preprocess, 274.8ms inference, 1.0ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 134.8ms
Speed: 2.5ms preprocess, 134.8ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 174.7ms
Speed: 2.8ms preprocess, 174.7ms inference, 1.1ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 149.8ms
Speed: 3.1ms preprocess, 149.8ms inference, 0.6ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 156.8ms
Speed: 2.7ms preprocess, 156.8ms inference, 0.6ms postprocess per image at